# Modellbewertung: gute und schwächere Beispiele

Erzeugt die kombinierte Beispielabbildung für gut und schwächer bewertete Modellannotationen.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while PROJECT_ROOT != PROJECT_ROOT.parent and not (PROJECT_ROOT / "README.md").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
__file__ = str(PROJECT_ROOT / "exports" / "notebooks" / "03_modellbewertung_textbeispiele_notebook.py")
print(f"Projektordner: {PROJECT_ROOT}")


## Code


In [ ]:
from __future__ import annotations

import csv
from pathlib import Path

import pandas as pd
from PIL import Image, ImageDraw, ImageFont, ImageOps


ROOT = Path(__file__).resolve().parents[2]
OUT_DIR = ROOT / "exports" / "examples" / "figures"
WORK_FIGURE_DIR = ROOT / "exports" / "examples" / "figure_variants"
WORK_META_DIR = ROOT / "exports" / "examples" / "metadata"
FRAME_DIR = ROOT / "exports" / "examples" / "frames"
METADATA_CSV = WORK_META_DIR / "screenshot_examples_metadata.csv"

TILE_W = 340
TILE_H = 604
LABEL_H = 220
GAP = 24
MARGIN = 36

BG = (248, 247, 242)
PANEL_BG = (255, 255, 255)
TEXT = (28, 31, 35)
MUTED = (78, 84, 93)
GOOD = (44, 125, 110)
WEAK = (150, 78, 72)

GOOD_EXAMPLES = [
    ("abb_kategorie_cuts_v02.png", 1, "Schnittdynamik"),
    ("abb_kategorie_camera_stability_v03.png", 1, "Kamerastabilität"),
    ("abb_kategorie_camera_distance_v01.png", 2, "Kameradistanz"),
    ("abb_kategorie_saturation_v01.png", 1, "Farbsättigung"),
    ("abb_kategorie_sharpness_v05.png", 1, "Schärfe"),
    ("abb_kategorie_face_orientation_v03.png", 1, "Gesichtsausrichtung"),
]

WEAK_EXAMPLES = [
    ("abb_kategorie_scene_v04.png", 3, "Szenenkontext"),
    ("abb_kategorie_skin_smoothness_v02.png", 3, "Hautglättung"),
    ("abb_kategorie_beauty_score_v02.png", 3, "Beauty Score"),
    ("abb_kategorie_body_pose_v01.png", 3, "Körperpose"),
]

COMBINED_GOOD_COUNT = 3
COMBINED_WEAK_COUNT = 3

def font(size: int, bold: bool = False) -> ImageFont.FreeTypeFont | ImageFont.ImageFont:
    candidates = [
        Path("C:/Windows/Fonts/arialbd.ttf" if bold else "C:/Windows/Fonts/arial.ttf"),
        Path("C:/Windows/Fonts/calibrib.ttf" if bold else "C:/Windows/Fonts/calibri.ttf"),
        Path("/System/Library/Fonts/Supplemental/Arial Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Arial.ttf"),
        Path("/System/Library/Fonts/Supplemental/Helvetica Bold.ttf" if bold else "/System/Library/Fonts/Supplemental/Helvetica.ttf"),
        Path("/usr/share/fonts/truetype/dejavu/DejaVuSans-Bold.ttf" if bold else "/usr/share/fonts/truetype/dejavu/DejaVuSans.ttf"),
    ]
    for path in candidates:
        if path.exists():
            return ImageFont.truetype(str(path), size)
    return ImageFont.load_default()


FONT_TITLE = font(18, bold=True)
FONT_SECTION = font(24, bold=True)
FONT_SMALL = font(13)


def draw_wrapped(
    draw: ImageDraw.ImageDraw,
    xy: tuple[int, int],
    text: str,
    max_width: int,
    fnt,
    fill=TEXT,
    max_lines: int = 3,
) -> int:
    words = str(text).split()
    lines: list[str] = []
    line = ""
    for word in words:
        candidate = f"{line} {word}".strip()
        if draw.textbbox((0, 0), candidate, font=fnt)[2] <= max_width or not line:
            line = candidate
        else:
            lines.append(line)
            line = word
    if line:
        lines.append(line)

    x, y = xy
    line_height = draw.textbbox((0, 0), "Ag", font=fnt)[3] + 6
    for visible_line in lines[:max_lines]:
        draw.text((x, y), visible_line, font=fnt, fill=fill)
        y += line_height
    return y


def selected_frame_path(row: pd.Series) -> Path:
    stem = Path(row["figure"]).stem
    path = FRAME_DIR / f"{stem}_{int(row['position']):02d}_{row['type']}_{row['video_id']}.jpeg"
    if not path.exists():
        raise FileNotFoundError(path)
    return path


def rows_for(selection: list[tuple[str, int, str]]) -> pd.DataFrame:
    metadata = pd.read_csv(METADATA_CSV, dtype={"video_id": str})
    selected_rows = []
    for figure, position, label in selection:
        row = metadata[(metadata["figure"] == figure) & (metadata["position"] == position)]
        if row.empty:
            raise ValueError(f"No metadata row for {figure} position {position}")
        selected = row.iloc[0].copy()
        selected["example_label"] = label
        selected["frame_path"] = selected_frame_path(selected)
        selected_rows.append(selected)
    return pd.DataFrame(selected_rows)


def draw_tile(
    canvas: Image.Image,
    draw: ImageDraw.ImageDraw,
    row: pd.Series,
    x: int,
    y: int,
    stripe: tuple[int, int, int],
) -> None:
    image = Image.open(row["frame_path"]).convert("RGB")
    image = ImageOps.fit(image, (TILE_W, TILE_H), method=Image.Resampling.LANCZOS, centering=(0.5, 0.5))
    canvas.paste(image, (x, y))

    draw.rectangle((x, y + TILE_H, x + TILE_W, y + TILE_H + LABEL_H), fill=PANEL_BG)
    draw.rectangle((x, y + TILE_H, x + 8, y + TILE_H + LABEL_H), fill=stripe)

    label_x = x + 18
    label_y = y + TILE_H + 8
    title = f"{row['example_label']} | {row['type']} | id={str(row['video_id'])[-6:]}"
    label_y = draw_wrapped(draw, (label_x, label_y), title, TILE_W - 28, FONT_TITLE, fill=TEXT, max_lines=1)
    for raw_line in str(row["subtitle"]).splitlines():
        label_y = draw_wrapped(draw, (label_x, label_y + 4), raw_line, TILE_W - 28, FONT_SMALL, fill=MUTED, max_lines=4)


def render(path: Path, rows: pd.DataFrame, columns: int, stripe: tuple[int, int, int]) -> None:
    row_count = (len(rows) + columns - 1) // columns
    width = MARGIN * 2 + columns * TILE_W + (columns - 1) * GAP
    height = MARGIN * 2 + row_count * (TILE_H + LABEL_H) + (row_count - 1) * GAP
    canvas = Image.new("RGB", (width, height), BG)
    draw = ImageDraw.Draw(canvas)

    for idx, (_, row) in enumerate(rows.iterrows()):
        grid_row = idx // columns
        grid_col = idx % columns
        x = MARGIN + grid_col * (TILE_W + GAP)
        y = MARGIN + grid_row * (TILE_H + LABEL_H + GAP)
        draw_tile(canvas, draw, row, x, y, stripe)

    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path, quality=95)


def render_combined(path: Path, good_rows: pd.DataFrame, weak_rows: pd.DataFrame) -> None:
    good_rows = good_rows.head(COMBINED_GOOD_COUNT).copy()
    weak_rows = weak_rows.head(COMBINED_WEAK_COUNT).copy()
    columns = max(len(good_rows), len(weak_rows))
    width = MARGIN * 2 + columns * TILE_W + (columns - 1) * GAP
    section_h = 42
    row_h = TILE_H + LABEL_H
    height = MARGIN * 2 + section_h * 2 + row_h * 2 + GAP
    canvas = Image.new("RGB", (width, height), BG)
    draw = ImageDraw.Draw(canvas)

    y = MARGIN
    draw.text((MARGIN, y), "Gut bewertete Modellannotationen", font=FONT_SECTION, fill=GOOD)
    y += section_h
    for idx, (_, row) in enumerate(good_rows.iterrows()):
        x = MARGIN + idx * (TILE_W + GAP)
        draw_tile(canvas, draw, row, x, y, GOOD)

    y += row_h + GAP
    draw.text((MARGIN, y), "Schwächer bewertete Modellannotationen", font=FONT_SECTION, fill=WEAK)
    y += section_h
    weak_width = len(weak_rows) * TILE_W + (len(weak_rows) - 1) * GAP
    start_x = MARGIN + (width - MARGIN * 2 - weak_width) // 2
    for idx, (_, row) in enumerate(weak_rows.iterrows()):
        x = start_x + idx * (TILE_W + GAP)
        draw_tile(canvas, draw, row, x, y, WEAK)

    path.parent.mkdir(parents=True, exist_ok=True)
    canvas.save(path, quality=95)


def write_metadata(path: Path, rows: pd.DataFrame) -> None:
    fieldnames = ["figure", "position", "video_id", "type", "example_label", "source_figure", "source_position", "subtitle", "frame_path"]
    with path.open("w", newline="", encoding="utf-8-sig") as fh:
        writer = csv.DictWriter(fh, fieldnames=fieldnames)
        writer.writeheader()
        for pos, (_, row) in enumerate(rows.iterrows(), start=1):
            writer.writerow(
                {
                    "figure": path.name.replace("_metadata.csv", ".png"),
                    "position": pos,
                    "video_id": row["video_id"],
                    "type": row["type"],
                    "example_label": row["example_label"],
                    "source_figure": row["figure"],
                    "source_position": row["position"],
                    "subtitle": row["subtitle"],
                    "frame_path": Path(row["frame_path"]).relative_to(ROOT),
                }
            )


def main() -> None:
    good_rows = rows_for(GOOD_EXAMPLES)
    weak_rows = rows_for(WEAK_EXAMPLES)

    good_png = WORK_FIGURE_DIR / "abb_modellbewertung_gute_beispiele.png"
    weak_png = WORK_FIGURE_DIR / "abb_modellbewertung_schwaechere_beispiele.png"
    combined_png = OUT_DIR / "abb_modellbewertung_gut_vs_schwaecher_beispiele.png"
    render(good_png, good_rows, columns=3, stripe=GOOD)
    render(weak_png, weak_rows, columns=4, stripe=WEAK)
    render_combined(combined_png, good_rows, weak_rows)

    WORK_META_DIR.mkdir(parents=True, exist_ok=True)
    write_metadata(WORK_META_DIR / "abb_modellbewertung_gute_beispiele_metadata.csv", good_rows)
    write_metadata(WORK_META_DIR / "abb_modellbewertung_schwaechere_beispiele_metadata.csv", weak_rows)
    combined_rows = pd.concat(
        [
            good_rows.head(COMBINED_GOOD_COUNT).assign(example_group="gut_bewertet"),
            weak_rows.head(COMBINED_WEAK_COUNT).assign(example_group="schwaecher_bewertet"),
        ],
        ignore_index=True,
    )
    write_metadata(OUT_DIR / "abb_modellbewertung_gut_vs_schwaecher_beispiele_metadata.csv", combined_rows)
    print(f"Created {good_png}")
    print(f"Created {weak_png}")
    print(f"Created {combined_png}")



## Ausführung


In [ ]:
main()
